# 任务五：基于双层LSTM的Seq2Seq机器翻译（德语→英语）
## 实验目标
1. 搭建 Encoder-Decoder 序列到序列模型
2. 使用双层LSTM实现德语到英语的端到端翻译
3. 采用Teacher Forcing策略加速训练，交叉熵作为损失函数
4. 完成数据预处理、模型构建、训练与推理全流程

代码解释：
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
作用：修改环境变量 HF_ENDPOINT，将 Hugging Face Hub（模型和数据集的官方托管平台）的默认访问端点替换为国内镜像源 https://hf-mirror.com。
原因：国内网络直接访问 Hugging Face 官方源（https://huggingface.co）可能不稳定或速度极慢。使用镜像源可以绕过网络限制，加速模型和数据集的下载。
影响：后续所有通过 datasets或 transformers库访问 Hugging Face Hub 的操作（如下载数据集、加载模型），都会自动从镜像源拉取资源，而非官方源。
dataset = datasets.load_dataset("bentrevett/multi30k")
作用：从 Hugging Face Datasets 库加载公开数据集 bentrevett/multi30k。
具体操作：
datasets.load_dataset()是 Hugging Face Datasets 库的核心函数，用于下载并加载指定的数据集。
第一个参数 "bentrevett/multi30k"是数据集的唯一标识符（格式为 用户名/数据集名），指向 Hugging Face Hub 上公开的 multi30k数据集（这是一个经典的机器翻译数据集，包含英语-德语的句子对）。
执行后，dataset变量会被赋值为一个 DatasetDict对象，通常包含训练集、验证集、测试集等子集，后续可通过 dataset["train"]、dataset["validation"]等方式访问。

## 1. 环境与依赖导入
导入PyTorch、spaCy等核心库，加载英语/德语预训练分词模型，完成基础环境初始化。

In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
import spacy
import datasets
import torchtext
import tqdm
import evaluate

In [4]:
seed = 1234

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

# 数据集 Dataset

### 数据集加载代码说明
1. `os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'`
将Hugging Face数据源切换为国内镜像，解决国内网络访问官方源超时、连接失败的问题，保障数据集正常下载与加载。
2. `dataset = datasets.load_dataset("bentrevett/multi30k")`
加载德英双语平行翻译数据集multi30k，该数据集是Seq2Seq机器翻译任务的标准数据集，包含训练集、验证集和测试集，用于模型训练与评估。

In [5]:
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
dataset = datasets.load_dataset("bentrevett/multi30k")

'[WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝试失败。' thrown while requesting HEAD https://huggingface.co/datasets/bentrevett/multi30k/resolve/4589883f3d09d4ef6361784e03f0ead219836469/multi30k.py
Retrying in 1s [Retry 1/5].
Using the latest cached version of the dataset since bentrevett/multi30k couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\DELL\.cache\huggingface\datasets\bentrevett___multi30k\default\0.0.0\4589883f3d09d4ef6361784e03f0ead219836469 (last modified on Thu Apr 16 19:49:08 2026).


## 2、运行下方的单元格。
你会看到数据集对象（一个DatasetDict）包含训练、验证和测试集，每个集合中的样本数量，以及每个集合中的特征（“en”和“de”）。


In [6]:
dataset

DatasetDict({
    train: Dataset({
        features: ['en', 'de'],
        num_rows: 29000
    })
    validation: Dataset({
        features: ['en', 'de'],
        num_rows: 1014
    })
    test: Dataset({
        features: ['en', 'de'],
        num_rows: 1000
    })
})

In [7]:
train_data, valid_data, test_data = (
    dataset["train"],
    dataset["validation"],
    dataset["test"],
)

## 3、运行下方的单元格。
我们可以索引每个数据集来查看单个示例。每个例子都有两个特征：“en”和“de”，是对应的英语和德语。


In [8]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.'}

接下来我们进行分词。英语/德语的分词较中文要直接，比如句子"good morning!会被分词为["good", "morning", "!"]序列。

下方的代码要成功安装en_core_web_sm和de_core_news_sm后才不会报错。

# 分词器 Tokenizers

In [9]:
en_nlp = spacy.load("en_core_web_sm")
de_nlp = spacy.load("de_core_news_sm")

## 4、运行下方的单元格。
我们可以使用.tokenizer方法调用每个spaCy模型的分词器，该方法接受字符串并返回Token对象序列。我们可以使用text属性从Token对象中获取字符串。


In [10]:
string = "What a lovely day it is today!"

[token.text for token in en_nlp.tokenizer(string)]

['What', 'a', 'lovely', 'day', 'it', 'is', 'today', '!']

## 5、添加一个Markdown单元格，在其中解释下方单元格的函数的作用。


### 函数作用
对单条德英平行语料进行**分词、长度截断、小写归一化、添加起止标记**预处理。
- 使用spaCy分别对英语、德语句子分词
- 限制句子最大长度，避免过长序列
- 统一转为小写，提升模型泛化能力
- 首尾添加 `<sos>` 起始符、`<eos>` 终止符，用于序列边界识别
- 返回分词后的英德token列表

In [11]:
def tokenize_example(example, en_nlp, de_nlp, max_length, lower, sos_token, eos_token):
    en_tokens = [token.text for token in en_nlp.tokenizer(example["en"])][:max_length]
    de_tokens = [token.text for token in de_nlp.tokenizer(example["de"])][:max_length]
    if lower:
        en_tokens = [token.lower() for token in en_tokens]
        de_tokens = [token.lower() for token in de_tokens]
    en_tokens = [sos_token] + en_tokens + [eos_token]
    de_tokens = [sos_token] + de_tokens + [eos_token]
    return {"en_tokens": en_tokens, "de_tokens": de_tokens}

## 6、添加一个Markdown单元格，在其中解释下方单元格出现的\<sos>和\<eos>的含义，以及map函数的作用。


### 特殊标记含义
- `<sos>`：**序列起始标记**，标识句子开始
- `<eos>`：**序列终止标记**，标识句子结束

### map函数作用
批量对数据集（train/valid/test）的**每一条样本**执行 `tokenize_example` 预处理函数，完成全量数据分词格式化，高效并行处理数据集。

In [12]:
max_length = 1_000
lower = True
sos_token = "<sos>"
eos_token = "<eos>"

fn_kwargs = {
    "en_nlp": en_nlp,
    "de_nlp": de_nlp,
    "max_length": max_length,
    "lower": lower,
    "sos_token": sos_token,
    "eos_token": eos_token,
}

train_data = train_data.map(tokenize_example, fn_kwargs=fn_kwargs)
valid_data = valid_data.map(tokenize_example, fn_kwargs=fn_kwargs)
test_data = test_data.map(tokenize_example, fn_kwargs=fn_kwargs)

## 7、运行下方的单元格
重新打印train_data\[0]，验证小写字符串列表以及序列标记的开始/结束符已被成功添加。


In [13]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>']}

# 词汇表 Vocabularies

下一个步骤是为源语言和目标语言构建词汇表，将词语映射为数字索引。比如"hello" = 1, "world" = 2, "bye" = 3, "hates" = 4。当向我们的模型提供文本数据时，我们使用词汇表作为look-up-table将字符串转换为标记，然后将标记转换为数字。“hello world”变成了“\[“hello”，“world”]”，然后变成了“\[1,2]”。

In [14]:
min_freq = 2
unk_token = "<unk>"
pad_token = "<pad>"

special_tokens = [
    unk_token,
    pad_token,
    sos_token,
    eos_token,
]

en_vocab = torchtext.vocab.build_vocab_from_iterator(
    train_data["en_tokens"],
    min_freq=min_freq,
    specials=special_tokens,
)

de_vocab = torchtext.vocab.build_vocab_from_iterator(
    train_data["de_tokens"],
    min_freq=min_freq,
    specials=special_tokens,
)

## 8、运行下方两个单元格
验证词汇表，分别打印英语词汇表和德语词汇表的前十个Token。


In [15]:
en_vocab.get_itos()[:10]

['<unk>', '<pad>', '<sos>', '<eos>', 'a', '.', 'in', 'the', 'on', 'man']

In [16]:
de_vocab.get_itos()[:10]

['<unk>', '<pad>', '<sos>', '<eos>', '.', 'ein', 'einem', 'in', 'eine', ',']

## 9、运行下方的单元格
使用get_stoi（stoi = "string to int "）方法获取指定的Token的索引。

In [17]:
en_vocab["the"]

7

In [18]:
assert en_vocab[unk_token] == de_vocab[unk_token]
assert en_vocab[pad_token] == de_vocab[pad_token]

unk_index = en_vocab[unk_token]
pad_index = en_vocab[pad_token]

In [19]:
en_vocab.set_default_index(unk_index)
de_vocab.set_default_index(unk_index)

词汇表的另一个有用特性是lookup_indices方法。它接受一个Token列表并返回一个索引列表。

## 10、运行下方的单元格
观察从Token列表到索引列表的转换。

In [20]:
tokens = ["i", "love", "watching", "crime", "shows"]
en_vocab.lookup_indices(tokens)

[956, 2169, 173, 0, 821]

对应的，lookup_tokens方法使用词汇表将索引列表转换回Token列表。

## 11、运行下方的单元格
观察从索引列表到Token列表的转换。


In [21]:
en_vocab.lookup_tokens(en_vocab.lookup_indices(tokens))

['i', 'love', 'watching', '<unk>', 'shows']

## 12、添加一个Markdown单元格，在其中解释为什么原本的"crime"被转换成了\<unk>。

### <unk> 含义与原因
- `<unk>` = unknown，**未知词标记**
- 原因：词汇表(vocab)构建时仅包含训练集中高频词，`crime` 未出现在词汇表中，因此被映射为 `<unk>` 占位符，保证模型可正常输入。

## 13、添加一个Markdown单元格，在其中解释下方两个单元格中代码的作用。


### 函数作用
完成**文本token向数字索引的映射**（数值化），是模型输入的必要步骤。
- 通过词汇表 `lookup_indices` 方法，将分词后的字符串token转为整型id
- 分别处理英语、德语序列
- 返回数值化后的id序列，供LSTM模型张量计算

In [22]:
def numericalize_example(example, en_vocab, de_vocab):
    en_ids = en_vocab.lookup_indices(example["en_tokens"])
    de_ids = de_vocab.lookup_indices(example["de_tokens"])
    return {"en_ids": en_ids, "de_ids": de_ids}

In [23]:
fn_kwargs = {"en_vocab": en_vocab, "de_vocab": de_vocab}

train_data = train_data.map(numericalize_example, fn_kwargs=fn_kwargs)
valid_data = valid_data.map(numericalize_example, fn_kwargs=fn_kwargs)
test_data = test_data.map(numericalize_example, fn_kwargs=fn_kwargs)

## 14、运行下方的单元格
重新打印train_data\[0]，验证"en_ids" and "de_ids"被成功添加。


In [24]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>'],
 'en_ids': [2, 16, 24, 15, 25, 778, 17, 57, 80, 202, 1312, 5, 3],
 'de_ids': [2, 18, 26, 253, 30, 84, 20, 88, 7, 15, 110, 7647, 3171, 4, 3]}

Dataset类为我们处理的另一件事是将features转换为正确的类型。每个例子中的索引目前都是基本的Python整数。然而，为了在PyTorch中使用它们，它们需要转换为PyTorch张量。with_format方法将columns参数转换为给定的类型。这里，我们指定类型为“torch”，columns为“en_ids”和“de_ids”（我们想要转换为PyTorch张量的features）。默认情况下，with_format将删除任何不在传递给列的features列表中的features。我们希望保留这些features，这可以通过output_all_columns=True来实现。

In [25]:
data_type = "torch"
format_columns = ["en_ids", "de_ids"]

train_data = train_data.with_format(
    type=data_type, columns=format_columns, output_all_columns=True
)

valid_data = valid_data.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

test_data = test_data.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

## 15、运行下方的单元格
重新打印train_data[0]，验证“en_ids”和“de_ids”特征被转换为了张量。

In [26]:
train_data[0]

{'en_ids': tensor([   2,   16,   24,   15,   25,  778,   17,   57,   80,  202, 1312,    5,
            3]),
 'de_ids': tensor([   2,   18,   26,  253,   30,   84,   20,   88,    7,   15,  110, 7647,
         3171,    4,    3]),
 'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>']}

# Data Loaders

数据准备的最后一步是创建Data Loaders。可以对它们进行迭代以返回一批数据，每一批数据都是一个字典，其中包含数字化的英语和德语句子作为PyTorch张量。

## 16、添加一个Markdown单元格，在其中解释下方两个单元格中的函数的作用。

### 函数作用
1. **get_collate_fn**：自定义批次处理函数
   - 提取批次中英德id序列
   - 使用 `pad_sequence` 对**不等长序列做padding补零**，统一批次长度
   - 返回规整后的批次张量

2. **get_data_loader**：构建PyTorch数据加载器
   - 封装数据集、批次大小、自定义padding函数
   - 支持训练集打乱(shuffle)，批量迭代读取数据，适配模型训练

In [27]:
def get_collate_fn(pad_index):
    def collate_fn(batch):
        batch_en_ids = [example["en_ids"] for example in batch]
        batch_de_ids = [example["de_ids"] for example in batch]
        batch_en_ids = nn.utils.rnn.pad_sequence(batch_en_ids, padding_value=pad_index)
        batch_de_ids = nn.utils.rnn.pad_sequence(batch_de_ids, padding_value=pad_index)
        batch = {
            "en_ids": batch_en_ids,
            "de_ids": batch_de_ids,
        }
        return batch

    return collate_fn

In [28]:
def get_data_loader(dataset, batch_size, pad_index, shuffle=False):
    collate_fn = get_collate_fn(pad_index)
    data_loader = torch.utils.data.DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        collate_fn=collate_fn,
        shuffle=shuffle,
    )
    return data_loader

In [29]:
batch_size = 128

train_data_loader = get_data_loader(train_data, batch_size, pad_index, shuffle=True)
valid_data_loader = get_data_loader(valid_data, batch_size, pad_index)
test_data_loader = get_data_loader(test_data, batch_size, pad_index)

# 构建模型

我们将分三部分构建模型。编码器，解码器和封装编码器和解码器的seq2seq模型。

# 编码器 Encoder

### Encoder（编码器）说明
采用**双层LSTM**结构，输入德语序列经过Embedding词嵌入层转换为稠密向量；
LSTM逐时间步编码文本语义，最终输出的隐藏状态作为**上下文向量**，编码整句的全局语义信息，作为解码器的初始状态。

首先是编码器，它是一个2层的LSTM。

## 17、添加一个Markdown单元格，解释下方单元格中Encoder类的代码。
包括输入参数，核心组件（词嵌入层、LSTM层、Dropout层），forwad函数的处理流程，和输出。

### Encoder 编码器详解
1. **输入参数**
   - input_dim：源语言(德语)词汇表大小
   - embedding_dim：词嵌入向量维度
   - hidden_dim：LSTM隐藏层维度
   - n_layers：LSTM层数
   - dropout：dropout随机失活比例

2. **核心组件**
   - Embedding层：将单词id转为稠密词向量
   - 双层LSTM：提取序列时序特征，编码语义信息
   - Dropout层：防止过拟合，增强泛化能力

3. **forward流程**
   - 输入源序列张量 → 词嵌入 → dropout正则化
   - 送入LSTM完成时序编码
   - 输出LSTM最终隐藏状态hidden、细胞状态cell

4. **输出**
   hidden与cell作为**上下文语义向量**，传递给Decoder初始化

In [30]:
class Encoder(nn.Module):
    def __init__(self, input_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(input_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # src = [src length, batch size]
        embedded = self.dropout(self.embedding(src))
        # embedded = [src length, batch size, embedding dim]
        outputs, (hidden, cell) = self.rnn(embedded)
        # outputs = [src length, batch size, hidden dim * n directions]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # outputs are always from the top hidden layer
        return hidden, cell

# 解码器 Decoder

### Decoder（解码器）说明
解码器初始状态继承编码器的上下文向量，以`<sos>`起始符为输入；
采用**教师强制（Teacher Forcing）**训练策略，以一定概率输入真实标签词加速模型收敛；
逐时间步生成英语单词，最终输出完整翻译序列，以`<eos>`终止符结束。

接下来是解码器，它需要与编码器对齐，同样是一个2层的LSTM。

## 18、添加一个Markdown单元格，描述Decoder的工作流程。

### Decoder 解码器工作流程
1. 初始化：接收Encoder输出的hidden/cell作为初始状态，继承源语句语义
2. 单步输入：接收上一时间步的单词id，升维后做词嵌入与dropout
3. LSTM解码：基于当前输入与历史状态，生成当前步隐藏状态
4. 全连接输出：将隐藏状态映射为目标词汇表维度，生成单词预测概率
5. 迭代生成：循环执行，逐词生成英语译文，直到输出<eos>终止
6. 输出：当前步预测结果、更新后的hidden/cell，供下一步解码

In [31]:
class Decoder(nn.Module):
    def __init__(self, output_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(output_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        # input = [batch size]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # n directions in the decoder will both always be 1, therefore:
        # hidden = [n layers, batch size, hidden dim]
        # context = [n layers, batch size, hidden dim]
        input = input.unsqueeze(0)
        # input = [1, batch size]
        embedded = self.dropout(self.embedding(input))
        # embedded = [1, batch size, embedding dim]
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        # output = [seq length, batch size, hidden dim * n directions]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # seq length and n directions will always be 1 in this decoder, therefore:
        # output = [1, batch size, hidden dim]
        # hidden = [n layers, batch size, hidden dim]
        # cell = [n layers, batch size, hidden dim]
        prediction = self.fc_out(output.squeeze(0))
        # prediction = [batch size, output dim]
        return prediction, hidden, cell

# Seq2Seq

## 19、添加一个Markdown单元格，解释下方单元格中Seq2Seq类的代码。
包括forward函数的流程，以及teacher forcing机制。

### Seq2Seq 模型整体流程
1. 初始化：绑定Encoder、Decoder、设备，校验隐藏层与层数维度一致
2. forward前向传播
   - 编码器编码德语源序列，输出上下文状态
   - 解码器以<sos>为起始输入，逐时间步生成译文
   - 存储每一步预测结果，构建输出张量
3. 输出：完整目标序列预测张量

### Teacher Forcing 机制
- 定义：训练时**按设定概率使用真实标签**作为解码器下一输入，而非模型预测值
- 作用：解决训练初期预测不准导致的误差累积，**加速模型收敛**
- 实现：随机数判断是否启用，概率由 teacher_forcing_ratio 控制

In [32]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        assert (
            encoder.hidden_dim == decoder.hidden_dim
        ), "Hidden dimensions of encoder and decoder must be equal!"
        assert (
            encoder.n_layers == decoder.n_layers
        ), "Encoder and decoder must have equal number of layers!"

    def forward(self, src, trg, teacher_forcing_ratio):
        # src = [src length, batch size]
        # trg = [trg length, batch size]
        # teacher_forcing_ratio is probability to use teacher forcing
        # e.g. if teacher_forcing_ratio is 0.75 we use ground-truth inputs 75% of the time
        batch_size = trg.shape[1]
        trg_length = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim
        # tensor to store decoder outputs
        outputs = torch.zeros(trg_length, batch_size, trg_vocab_size).to(self.device)
        # last hidden state of the encoder is used as the initial hidden state of the decoder
        hidden, cell = self.encoder(src)
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # first input to the decoder is the <sos> tokens
        input = trg[0, :]
        # input = [batch size]
        for t in range(1, trg_length):
            # insert input token embedding, previous hidden and previous cell states
            # receive output tensor (predictions) and new hidden and cell states
            output, hidden, cell = self.decoder(input, hidden, cell)
            # output = [batch size, output dim]
            # hidden = [n layers, batch size, hidden dim]
            # cell = [n layers, batch size, hidden dim]
            # place predictions in a tensor holding predictions for each token
            outputs[t] = output
            # decide if we are going to use teacher forcing or not
            teacher_force = random.random() < teacher_forcing_ratio
            # get the highest predicted token from our predictions
            top1 = output.argmax(1)
            # if teacher forcing, use actual next token as next input
            # if not, use predicted token
            input = trg[t] if teacher_force else top1
            # input = [batch size]
        return outputs

# 模型训练

模型初始化

## 训练参数与策略配置
1. 损失函数：交叉熵损失，计算预测序列与真实序列差异
2. 优化器：Adam优化器，更新模型参数
3. 训练策略：Teacher Forcing，按概率输入真实标签，加速模型收敛

## 20、添加注释
分别将“# 编码器初始化”，“# 解码器初始化”，“# Seq2Seq模型整合”这三行注释加到下方单元格中正确的位置

In [33]:
input_dim = len(de_vocab)
output_dim = len(en_vocab)
encoder_embedding_dim = 256
decoder_embedding_dim = 256
hidden_dim = 512
n_layers = 2
encoder_dropout = 0.5
decoder_dropout = 0.5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 编码器初始化
encoder = Encoder(
    input_dim,
    encoder_embedding_dim,
    hidden_dim,
    n_layers,
    encoder_dropout,
)

# 解码器初始化
decoder = Decoder(
    output_dim,
    decoder_embedding_dim,
    hidden_dim,
    n_layers,
    decoder_dropout,
)

# Seq2Seq模型整合
model = Seq2Seq(encoder, decoder, device).to(device)

权重初始化

In [34]:
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)


model.apply(init_weights)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(7853, 256)
    (rnn): LSTM(256, 512, num_layers=2, dropout=0.5)
    (dropout): Dropout(p=0.5, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(5893, 256)
    (rnn): LSTM(256, 512, num_layers=2, dropout=0.5)
    (fc_out): Linear(in_features=512, out_features=5893, bias=True)
    (dropout): Dropout(p=0.5, inplace=False)
  )
)

In [35]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print(f"The model has {count_parameters(model):,} trainable parameters")

The model has 13,898,501 trainable parameters


优化器 optimizer

In [36]:
optimizer = optim.Adam(model.parameters())

损失函数 Loss Function

In [37]:
criterion = nn.CrossEntropyLoss(ignore_index=pad_index)

Training Loop:

## 21、给下方单元格中的代码逐行加注释

In [38]:
def train_fn(
    model, data_loader, optimizer, criterion, clip, teacher_forcing_ratio, device
):
    model.train()  # 设置模型为训练模式，启用dropout/batchnorm
    epoch_loss = 0  # 初始化本轮epoch总损失
    for i, batch in enumerate(data_loader):  # 遍历数据加载器，逐批次训练
        src = batch["de_ids"].to(device)  # 加载德语源序列，移至GPU/CPU
        trg = batch["en_ids"].to(device)  # 加载英语目标序列，移至GPU/CPU
        # src = [src length, batch size]
        # trg = [trg length, batch size]
        optimizer.zero_grad()  # 清空梯度，避免梯度累积
        output = model(src, trg, teacher_forcing_ratio)  # 前向传播，获取预测结果
        # output = [trg length, batch size, trg vocab size]
        output_dim = output.shape[-1]  # 获取目标词汇表维度
        output = output[1:].view(-1, output_dim)  # 去除<sos>，展平适配损失计算
        # output = [(trg length - 1) * batch size, trg vocab size]
        trg = trg[1:].view(-1)  # 去除<sos>，展平真实标签
        # trg = [(trg length - 1) * batch size]
        loss = criterion(output, trg)  # 计算交叉熵损失
        loss.backward()  # 反向传播，计算梯度
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)  # 梯度裁剪，防止梯度爆炸
        optimizer.step()  # 更新模型参数
        epoch_loss += loss.item()  # 累加批次损失
    return epoch_loss / len(data_loader)  # 返回本轮平均损失

Evaluation Loop:

In [39]:
def evaluate_fn(model, data_loader, criterion, device):
    model.eval()  # 设置模型为评估模式，关闭dropout/batchnorm
    epoch_loss = 0  # 初始化本轮评估总损失
    with torch.no_grad():  # 关闭梯度计算，加速推理+节省显存
        for i, batch in enumerate(data_loader):  # 遍历评估数据集
            src = batch["de_ids"].to(device)  # 加载德语源序列
            trg = batch["en_ids"].to(device)  # 加载英语目标序列
            # src = [src length, batch size]
            # trg = [trg length, batch size]
            output = model(src, trg, 0)  # 推理关闭teacher forcing，纯模型自回归预测
            # output = [trg length, batch size, trg vocab size]
            output_dim = output.shape[-1]  # 获取词汇表维度
            output = output[1:].view(-1, output_dim)  # 展平预测结果
            # output = [(trg length - 1) * batch size, trg vocab size]
            trg = trg[1:].view(-1)  # 展平真实标签
            # trg = [(trg length - 1) * batch size]
            loss = criterion(output, trg)  # 计算评估损失
            epoch_loss += loss.item()  # 累加损失
    return epoch_loss / len(data_loader)  # 返回平均评估损失

# 模型训练

##  模型训练执行
1. 批量加载数据，前向传播计算预测结果
2. 反向传播更新梯度，迭代优化模型参数
3. 记录训练损失，监控模型收敛效果

### 模型训练说明
1. 训练轮数`n_epochs=5`：降低计算量，满足作业基础要求；
2. `clip=1.0`：梯度裁剪，防止训练过程中梯度爆炸；
3. `teacher_forcing_ratio=0.5`：教师强制概率，平衡训练速度与模型泛化能力；
4. 采用交叉熵损失函数，自动保存验证损失最优的模型权重。

In [40]:
n_epochs = 5 # 因模型训练对计算资源要求较高，此处设立了5轮训练。
clip = 1.0
teacher_forcing_ratio = 0.5

best_valid_loss = float("inf")

for epoch in tqdm.tqdm(range(n_epochs)):
    train_loss = train_fn(
        model,
        train_data_loader,
        optimizer,
        criterion,
        clip,
        teacher_forcing_ratio,
        device,
    )
    valid_loss = evaluate_fn(
        model,
        valid_data_loader,
        criterion,
        device,
    )
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), "tut1-model.pt")
    print(f"\tTrain Loss: {train_loss:7.3f} | Train PPL: {np.exp(train_loss):7.3f}")
    print(f"\tValid Loss: {valid_loss:7.3f} | Valid PPL: {np.exp(valid_loss):7.3f}")

 20%|██████████▏                                        | 1/5 [00:27<01:50, 27.58s/it]

	Train Loss:   5.042 | Train PPL: 154.737
	Valid Loss:   4.992 | Valid PPL: 147.172


 40%|████████████████████▍                              | 2/5 [00:53<01:20, 26.73s/it]

	Train Loss:   4.439 | Train PPL:  84.696
	Valid Loss:   4.811 | Valid PPL: 122.886


 60%|██████████████████████████████▌                    | 3/5 [01:19<00:52, 26.44s/it]

	Train Loss:   4.161 | Train PPL:  64.107
	Valid Loss:   4.616 | Valid PPL: 101.045


 80%|████████████████████████████████████████▊          | 4/5 [01:45<00:26, 26.19s/it]

	Train Loss:   3.951 | Train PPL:  51.962
	Valid Loss:   4.446 | Valid PPL:  85.290


100%|███████████████████████████████████████████████████| 5/5 [02:10<00:00, 26.06s/it]

	Train Loss:   3.766 | Train PPL:  43.213
	Valid Loss:   4.325 | Valid PPL:  75.560


# 模型验证

## 模型推理与翻译测试
1. 关闭梯度计算，输入德语测试语句
2. 解码器逐词生成英语译文，自动终止于<eos>
3. 验证模型翻译效果，完成Seq2Seq任务闭环

In [41]:
model.load_state_dict(torch.load("tut1-model.pt"))

<All keys matched successfully>

### 翻译推理说明
调用自定义翻译函数，输入德语句子完成分词、编码、解码全流程；
因仅训练1轮，翻译效果有限属于正常现象；增加训练轮数可显著提升翻译准确率。

In [42]:
def translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
    max_output_length=25,
):
    model.eval()
    with torch.no_grad():
        if isinstance(sentence, str):
            tokens = [token.text for token in de_nlp.tokenizer(sentence)]
        else:
            tokens = [token for token in sentence]
        if lower:
            tokens = [token.lower() for token in tokens]
        tokens = [sos_token] + tokens + [eos_token]
        ids = de_vocab.lookup_indices(tokens)
        tensor = torch.LongTensor(ids).unsqueeze(-1).to(device)
        hidden, cell = model.encoder(tensor)
        inputs = en_vocab.lookup_indices([sos_token])
        for _ in range(max_output_length):
            inputs_tensor = torch.LongTensor([inputs[-1]]).to(device)
            output, hidden, cell = model.decoder(inputs_tensor, hidden, cell)
            predicted_token = output.argmax(-1).item()
            inputs.append(predicted_token)
            if predicted_token == en_vocab[eos_token]:
                break
        tokens = en_vocab.lookup_tokens(inputs)
    return tokens

In [43]:
sentence = test_data[0]["de"]
expected_translation = test_data[0]["en"]

sentence, expected_translation

('Ein Mann mit einem orangefarbenen Hut, der etwas anstarrt.',
 'A man in an orange hat starring at something.')

In [44]:
translation = translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
)

# 22、运行下方单元格，得到测试集第0个索引的翻译
因为epoch只进行了一轮，不会有好的效果的翻译。
感兴趣的同学可自行增加训练轮数，观察loss和翻译质量的变化。

In [45]:
translation

['<sos>',
 'a',
 'man',
 'in',
 'a',
 'black',
 'shirt',
 'is',
 'a',
 'a',
 '.',
 '<eos>']

# 实验总结
1. 成功搭建双层LSTM的Encoder-Decoder翻译模型
2. 完成文本分词、词汇映射、序列格式化全流程预处理
3. Teacher Forcing策略有效提升训练收敛速度
4. 模型可实现德语到英语的基础序列翻译，达成任务目标